# Painel de séries: várias séries no mesmo experimento

Com um painel, as candidatas são comparadas em **todas** as séries ao mesmo tempo:

1. Cada série tem seu próprio **teste selado**, suas origens de validação e sua escala de MASE.
2. O **ranking** usa a melhor configuração de cada estrutura **em cada série** e tira a média dos postos entre as séries (convenção v1, ADR-0004).
3. A **seleção incremental** junta as unidades `série:dobra` de todas as séries: ganho mediano e taxa de vitórias são calculados sobre o conjunto.
4. A **finalização** reajusta cada série com a configuração escolhida para ela e devolve um `PanelForecaster`.

Um híbrido que ajuda muito em uma série e atrapalha em outra pode ser rejeitado no conjunto. Isso é intencional: a decisão vale para o painel.

In [ ]:
import autohybts

print("AutoHybTS", autohybts.__version__)

## 1. Montar o painel

Três formas equivalentes de entrada: um dicionário `{id: pd.Series}`, um `PanelFrame` ou uma tabela longa com as colunas nomeadas explicitamente.

In [ ]:
from autohybts import AutoHybridExperiment, PanelFrame
from autohybts.datasets import get_data

series = {
    "loja_a": get_data("synthetic_daily", n=400, seed=1),
    "loja_b": get_data("synthetic_daily", n=400, seed=2, phi=0.2),
    "loja_c": get_data("white_noise_daily", n=400, seed=3),
}
panel = PanelFrame.from_dict(series)
long = panel.to_long()
long.head()

## 2. Experimento

A mesma chamada de uma série única. Para a tabela longa, informe `series_col`, `date_col` e `target`.

In [ ]:
exp = AutoHybridExperiment(
    fh=14,
    session_id=42,
    fold=3,
    base_models=["naive", "seasonal_naive", "linear", "knn"],
    correctors=["linear"],
    residual_windows=[7],
).fit(long, series_col="series_id", date_col="timestamp", target="value")
exp.splits.head(8)

In [ ]:
exp.data_profile()

In [ ]:
exp.plot_data("series", series_id="loja_c")

## 3. Busca, ranking entre séries e seleção

In [ ]:
result = exp.compare_models()
result.leaderboard.head(10)

Histórico de decisões: `median_gain` e `win_rate` vêm das unidades de **todas** as séries (`n_units`).

In [ ]:
result.decisions

Configuração da estrutura campeã em cada série (a melhor configuração por série):

In [ ]:
{sid: spec.label for sid, spec in result.selection.champion_per_series.items()}

In [ ]:
exp.plot_model(result.best, "cv_forecasts", series_id="loja_a")

## 4. Validação cruzada temporal no painel

As dobras são criadas por série; as unidades da comparação são `série:dobra`.

In [ ]:
cv = exp.cross_validate(models=["naive", "seasonal_naive", "linear", result.best], folds=4)
cv.summary

In [ ]:
cv.plot("boxplot")

## 5. Finalizar, prever e avaliar no teste selado

Cada série é reajustada com a sua configuração até o fim do seu histórico de desenvolvimento. A avaliação externa traz uma linha por série e a média; ela só pode ser feita uma vez por modelo.

In [ ]:
model = exp.finalize_model(result.best)
model

In [ ]:
exp.predict_model(model).predictions.groupby("series_id").head(3)

In [ ]:
exp.evaluate_external(model).metrics

## 6. Salvar e carregar

In [ ]:
import tempfile

from autohybts import load_model, save_model

with tempfile.TemporaryDirectory() as folder:
    path = save_model(model, folder)
    loaded = load_model(path, trusted=True)
loaded.predict(3)